In [1]:
print('Hello World')

Hello World


In [47]:
import pandas as pd
import sqlalchemy
from sqlalchemy import create_engine, text

### Load the datasets

In [39]:
# load customers file
customers = pd.read_csv(r"C:\Users\Admin\Desktop\20260601-DE5M5\data\03_Library SystemCustomers.csv")
customers

,Customer ID,Customer Name
0,1.0,Jane Doe
1,2.0,John Smith
2,3.0,Dan Reeves
3,NaN,NaN
4,5.0,William Holden
5,6.0,Jaztyn Forest
6,7.0,Jackie Irving
7,8.0,Matthew Stirling
8,9.0,Emory Ted


In [40]:
# load bookings file
library_log = pd.read_csv(r"C:\Users\Admin\Desktop\20260601-DE5M5\data\03_Library Systembook.csv")
library_log

,Id,Books,Book checkout,Book Returned,Days allowed to borrow,Customer ID
0,1.0,Catcher in the Rye,"""20/02/2023""",25/02/2023,2 weeks,1.0
1,2.0,Lord of the rings the two towers,"""24/03/2023""",21/03/2023,2 weeks,2.0
2,3.0,Lord of the rings the return of the kind,"""29/03/2023""",25/03/2023,2 weeks,3.0
3,4.0,The hobbit,"""02/04/2023""",25/03/2023,2 weeks,4.0
4,5.0,Dune,"""02/04/2023""",25/03/2023,2 weeks,5.0
...,...,...,...,...,...,...
109,NaN,NaN,NaN,NaN,NaN,NaN
110,NaN,NaN,NaN,NaN,NaN,NaN
111,NaN,NaN,NaN,NaN,NaN,NaN
112,NaN,NaN,NaN,NaN,NaN,NaN


### Clean the datasets

In [41]:
#-------- customers csv  -----------------------

# count customers length
initial_cust_count = len(customers)

# drop completely null rows
customers = customers.dropna(how="all")

# ID column to integers
customers["Customer ID"] = customers["Customer ID"].astype(int)

C:\Users\Admin\AppData\Local\Temp\ipykernel_7356\2754799014.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  customers["Customer ID"] = customers["Customer ID"].astype(int)


In [42]:
#-------- library log csv  -----------------------

# initial_cust_count 
initial_log_count  = len(library_log)

# drop completely null rows
library_log = library_log.dropna(how="all")

# ID column to integers
customers["Customer ID"] = customers["Customer ID"].astype(int)

# drop duplicates
library_log = library_log.drop_duplicates(subset=["Books", "Book checkout", "Book Returned", "Customer ID"])

# drop if main info is missing
library_log = library_log.dropna(subset=["Books", "Customer ID"])

# Clean Text String Columns (strip quotes and trailing spaces)
library_log["Books"] = library_log["Books"].str.strip()
library_log["Book checkout"] = (library_log["Book checkout"].astype(str).str.replace('"', ""))

# Rigorous Date Validation & Parsing errors='coerce' turns invalid dates (like May 32nd) into NaT (Not a Time)
library_log["Book checkout"] = pd.to_datetime(library_log["Book checkout"], format="%d/%m/%Y", errors="coerce")
library_log["Book Returned"] = pd.to_datetime(library_log["Book Returned"], format="%d/%m/%Y", errors="coerce")

# unrealistic dates
library_log = library_log[(library_log["Book checkout"] <= pd.Timestamp("2026-06-01")) & (library_log["Book checkout"] <= library_log["Book Returned"])]



C:\Users\Admin\AppData\Local\Temp\ipykernel_7356\2634385746.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  customers["Customer ID"] = customers["Customer ID"].astype(int)


In [43]:
library_log

,Id,Books,Book checkout,Book Returned,Days allowed to borrow,Customer ID
0,1.0,Catcher in the Rye,2023-02-20,2023-02-25,2 weeks,1.0
5,6.0,Little Women,2023-04-02,2023-05-01,2 weeks,1.0
8,9.0,Catch 22,2023-04-15,2023-04-16,2 weeks,7.0
9,10.0,Animal Farm,2023-04-20,2023-04-24,2 weeks,2.0
10,11.0,1984,2023-04-23,2023-04-27,2 weeks,8.0
12,13.0,East of Eden,2023-04-30,2023-05-05,2 weeks,2.0
13,14.0,America Is in the Heart,2023-05-01,2023-05-07,2 weeks,3.0
14,15.0,Wuthering Heights,2023-05-01,2023-05-10,2 weeks,9.0
15,16.0,Dark Tales,2023-05-15,2023-06-01,2 weeks,2.0
17,18.0,Les Miserables,2023-06-03,2023-06-07,2 weeks,5.0


### Data engineering metrics

In [44]:
# Measure at least 1 data engineering metric in the data cleaning process (ie dropped rows)

dropped_customers = initial_cust_count - len(customers)
dropped_logs = initial_log_count - len(library_log)
log_data_integrity_rate = (len(library_log) / initial_log_count) * 100

print(f"Metrics - Customers Dropped Rows: {dropped_customers}")
print(f"Metrics - Library Log Dropped Rows: {dropped_logs}")
print(f"Metrics - Library Log Pipeline Success Rate: {log_data_integrity_rate:.2f}%\n")

Metrics - Customers Dropped Rows: 1
Metrics - Library Log Dropped Rows: 102
Metrics - Library Log Pipeline Success Rate: 10.53%



### Output to local csv fils

In [46]:
customers.to_csv("cleaned_customers.csv", index=False)
library_log.to_csv("cleaned_library_log.csv", index=False)

### Output the cleaned files into the local SSMS (use SQLAlchemy); create a new DB for this.

In [50]:

SERVER_NAME = "localhost"
NEW_DB_NAME = "LibrarySystem_DB"

master_url = f"mssql+pyodbc://@{SERVER_NAME}/master?trusted_connection=yes&driver=ODBC+Driver+17+for+SQL+Server"
target_url = f"mssql+pyodbc://@{SERVER_NAME}/{NEW_DB_NAME}?trusted_connection=yes&driver=ODBC+Driver+17+for+SQL+Server"

master_engine = create_engine(master_url)

check_db_query = text(
    f"SELECT database_id FROM sys.databases WHERE name = '{NEW_DB_NAME}'"
)

with master_engine.connect() as conn:
    # Set execution isolation level to execute outside an active transaction block
    conn = conn.execution_options(isolation_level="AUTOCOMMIT")
    result = conn.execute(check_db_query).fetchone()

    if not result:
        print(f"   Database '{NEW_DB_NAME}' not found. Spawning new DB...")
        conn.execute(text(f"CREATE DATABASE {NEW_DB_NAME}"))
        print(f"   Database '{NEW_DB_NAME}' successfully instantiated.")
    else:
        print(f"   Target database '{NEW_DB_NAME}' already exists. Skipping...")


target_engine = create_engine(target_url)

# to_sql drops and recreates tables automatically if they exist
customers.to_sql(
    name="Dim_Customers", con=target_engine, if_exists="replace", index=False
)
print("   Successfully wrote 'Dim_Customers' to SSMS.")

library_log.to_sql(
    name="Fact_LibraryLogs",
    con=target_engine,
    if_exists="replace",
    index=False,
)
print("   Successfully wrote 'Fact_LibraryLogs' to SSMS.")

print("\nPipeline complete!")

   Target database 'LibrarySystem_DB' already exists. Skipping...
   Successfully wrote 'Dim_Customers' to SSMS.
   Successfully wrote 'Fact_LibraryLogs' to SSMS.

Pipeline complete!
